# Интеллектуальный агент анализа конкурентной продукции

> На основе Привет Интеллектуальная система анализа конкурентной продукции на основе платформы Agents
> 
> - Автоматический сбор информации о конкурентных продуктах
> - Многомерный сравнительный анализ
> - Создавайте профессиональные отчеты

## Информация об авторе
- **Имя**: czxgg0630
- **GitHub**: [@czxgg0630](https://github.com/czxgg0630)
- **дата**: 2026-04-09

# Часть 2: Конфигурация среды

In [1]:
# Установить зависимости
!pip install -q hello-agents[all]

In [2]:
# Импортируйте необходимые библиотеки
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from hello_agents.tools.builtin.search_tool import SearchTool
from typing import Dict, Any, List
import os
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:8800'  # Адрес прокси
from dotenv import load_dotenv

# Загрузить переменные среды
load_dotenv()

True

# Часть 3: Определение инструмента

In [3]:
class CompetitiveInfoSearchTool(Tool):
    """Инструмент поиска информации о конкурентных продуктах - Используйте настоящий API поиска"""
    
    def __init__(self):
        super().__init__(
            name="competitive_info_search",
            description="Поиск информации о продукте конкурента: функции, цены, стратегия"
        )
        # Чтобы инициализировать встроенный инструмент поиска, используйте Tavily задняя часть
        self.search = SearchTool(backend="tavily")
    
    def get_parameters(self) -> List[ToolParameter]:
        """Получить определение параметров инструмента"""
        return [
            ToolParameter(
                name="product_name",
                type="string",
                description="Название конкурента для поиска",
                required=True
            )
        ]
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Инструмент выполнения, использующий реальный поиск"""
        product_name = parameters.get("product_name", "")
        print(f"🔍 Идет поиск {product_name} Информация о конкурентном продукте...")
        
        # Используйте настоящий API поиска
        try:
            search_query = f"{product_name} product features pricing pros cons 2024"
            result = self.search.run({
                "query": search_query,
                "max_results": 5
            })
            
            # Форматировать результаты поиска
            return f"""
【{product_name} результаты поиска】
{result}
"""
        except Exception as e:
            print(f"⚠️ Поиск не удался: {e}, используя резервные данные")
            # Если поиск не удался, возвращается сообщение с подсказкой.
            return f"""
【{product_name} информация】
- Ошибка поиска, проверьте сеть или API
- Название: {product_name}
- Дополните информацию вручную
"""


class DataProcessorTool(Tool):
    """инструменты обработки данных - Очистите и структурируйте данные о конкурентных продуктах"""
    
    def __init__(self):
        super().__init__(
            name="data_processor",
            description="Очистка данных и матрица сравнения"
        )
    
    def get_parameters(self) -> List[ToolParameter]:
        """Получить определение параметров инструмента"""
        return [
            ToolParameter(
                name="raw_data",
                type="string",
                description="Сырые собранные данные",
                required=True
            )
        ]
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Инструмент выполнения"""
        print("📊 Обработка и структурирование...")
        return """
【Результат структурирования】
1. ✓ Извлечены ключевые атрибуты
2. ✓ Единый формат данных
3. ✓ Каркас сравнения
"""


class ReportGeneratorTool(Tool):
    """Инструмент создания отчетов - Создавайте профессиональные отчеты по анализу конкурентной продукции"""
    
    def __init__(self):
        super().__init__(
            name="report_generator",
            description="Markdown-отчёт на основе анализа"
        )
    
    def get_parameters(self) -> List[ToolParameter]:
        """Получить определение параметров инструмента"""
        return [
            ToolParameter(
                name="analysis_data",
                type="string",
                description="Обработанные данные",
                required=True
            )
        ]
    
    def run(self, parameters: Dict[str, Any]) -> str:
        """Инструмент выполнения"""
        print("📝 Генерация отчёта...")
        return "# Отчёт конкурентного анализа\n\n## Резюме\nАнализ завершён..."

print("✅ Определены три основных инструмента (инструмент поиска использует реальный API)")

# Часть 4: Агентское строительство

In [4]:
# Создать LLM
llm = HelloAgentsLLM()

# Определение слов системных подсказок - Четко проинструктируйте LLM о том, как вызывать инструменты.
SYSTEM_PROMPT = """Ты эксперт по конкурентному анализу, умеешь системно анализировать несколько продуктов.

【Правила вызова инструментов — обязательны】

1. Для поиска вызывай competitive_info_search
2. Формат: {"product_name": "название"}
3. product_name — одно конкретное название, не пустое
4. Несколько продуктов — отдельный вызов на каждый

【Примеры】
- Notion: {"product_name": "Notion"}
- Obsidian: {"product_name": "Obsidian"}
- Logseq: {"product_name": "Logseq"}

【Рабочий процесс】
1. Извлеки все названия конкурентов
2. Поиск каждого через competitive_info_search
3. Обработка через data_processor
4. Отчёт через report_generator
5. Полный отчёт на основе поиска

【Важно】
- Не передавай пустой product_name
- Жди результат инструмента перед следующим шагом
- Отчёт только на реальных данных поиска, без выдумок"""

# Создайте главный агент управления
agent = SimpleAgent(
    name="Эксперт конкурентного анализа",
    llm=llm,
    system_prompt=SYSTEM_PROMPT
)

# Три основных инструмента
agent.add_tool(CompetitiveInfoSearchTool())
agent.add_tool(DataProcessorTool())
agent.add_tool(ReportGeneratorTool())

print("✅ Plan-and-Solve Агент анализа конкурентной продукции инициализирован.")
print("✅ Промпт с правилами инструментов настроен")

# Часть 5: Функциональная демонстрация

In [ ]:
# Пример 1: Базовый анализ конкурентного продукта
import time
from datetime import datetime

print("=" * 70)
print("📊 Пример 1: SimpleAgent Быстрый анализ конкурентной продукции")
print("=" * 70)

target_products = ["Hema", "Dingdong Maicai", "Sam's Club"]
print(f"\n🎯 Цели анализа: {', '.join(target_products)}")
print(f"⏰ время начала: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("-" * 70)

# Запуск анализа
start_time = time.time()
result = agent.run(
    f"Проведи глубокий сравнительный анализ: {', '.join(target_products)}."
)
elapsed_time = time.time() - start_time

# Красивый выходной макет
print("\n" + "=" * 70)
print("📋 Анализ завершен")
print("=" * 70)
print(f"⏱️  Общее время потрачено: {elapsed_time:.2f} Второй")
print(f"📝 Длина отчета: {len(result)} характер")
print("-" * 70)

# Показать сводку отчета
print("\n📄 Предварительный просмотр отчета (первые 1000 символов):")
print("-" * 70)
print(result[:1000] + "..." if len(result) > 1000 else result)
print("-" * 70)

# Сохранение в файл
output_filename = f"outputs/demo_result_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(f"# Отчёт конкурентного анализа\n\n")
    f.write(f"**Время анализа**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"**Продукты**: {', '.join(target_products)}\n\n")
    f.write(f"**Длительность**: {elapsed_time:.2f} с\n\n")
    f.write("---\n\n")
    f.write(result)

print(f"\n💾 Отчет сохранен в: {output_filename}")
print("=" * 70)

# Часть 6: Оценка эффективности

## Часть шестая: Оценка производительности и углубленный архитектурный анализ (SimpleAgent)

### 1. Архитектурный анализ SimpleAgent

SimpleAgent использовать **Монорельс/ReAct парадигма**, который в настоящее время является наиболее распространенным один из способов реализации Agent.

| Критерий | SimpleAgent | Пояснение |
|---------|-----------------|------|
| **Логическая сложность переноса** | ★★★ средний низкий | Полагается на механизм внимания LLM для одновременного"понимать контекст"、"Выбрать инструмент"и"решить следующий шаг". анализировать 3 шт.конкурентОсуществимо.Анализ 10 Очень легко галлюцинировать или промахнуться. |
| **потребление контекста (Token)** | ★★ выше | Все промежуточные результаты, ошибки, цикл рассуждений（Thought/Action/Наблюдение) расположены в одном контекстном окне, их легко активировать. Context Window предел. |
| **Контролируемость и вмешательство** | ★★ состояние черного ящика | После запуска сложно прервать и изменить путь рассуждений. |
| **Сложность реализации** | ★★★★★ минималистский | Объем кода небольшой, его легко понять и отладить, он подходит для обучения начального уровня и быстрой проверки прототипа. |
| **Скорость ответа** | ★★★★★ быстро | Никаких дополнительных шагов по планированию не требуется, прямой ответ на ввод пользователя, низкая сквозная задержка. |

### 2. Стабильность выполнения и анализ аномалий (Stability & Error Handling)

**1. Риск тайм-аута сети (Network/Timeout Issues)**

Симптом: при SimpleAgent возможны сетевые исключения, информация стека указывает на базовыйсетьсчитывает содержимое ячейки A1 на листе1 в(ssl.py: read) и httpx。

Диагноз: синхронная блокировка (псевдо-зависание).Либо LLMиз API медленный ответ или Tavily Интерфейс поиска заблокирован/Если ток ограничен, то весь основной поток зависнет.

Предложения по улучшению: В производственной среде любые внешние API вызов（LLM или Поиск) необходимо настроить с жесткими timeout стратегии в сочетании с механизмами повторных попыток, такими как tenacity библиотека), чтобы предотвратить выход из строя всей системы из-за колебаний сети в отдельных точках.

**2. Защитное программирование (Defensive Programming)**

Текущий статус: CompetitiveInfoSearchTool Уже реализовано в try...except обработка исключений и резервный текст.

ценность：Даже при сбое поиска Agent получает явную"обратная связь об ошибке"，вместо падения — LLM может повторить или пропустить.

### 3. Оценка цепочки инструментов

### Текущий статус: PoC (Доказательство концепции) этап

| инструмент | текущая реализация | Направления улучшения |
|------|---------|---------|
| **SearchTool** | ✅ Tavily API | Кэш, несколько бэкендов |
| **DataProcessorTool** | ⚠️ Фиксированная строка | LLM для очистки данных |
| **ReportGeneratorTool** | ⚠️ Вернуть фиксированную строку | Отчёт из реальных данных, не LLM |

### ключевой вопросТекущий DataProcessorToolиReportGeneratorTool - это просто "Финт"——Agent Их вызывали, но возвращались жестко закодированные строки, а окончательный длинный отчет все еще был LLM минуя инструменты.

### 4. Сценарии SimpleAgent

| сценарий | пригодность | иллюстрировать |
|------|--------|------|
| Быстрое прототипирование | ★★★★★ | Код краток и легко повторяется. |
| До 3 конкурентов | ★★★★ | Умеренная сложность |
| 10+ конкурентов | ★★ | Галлюцинации, много токенов |
| Развертывание производственной среды | ★★ | Отсутствие отказоустойчивости и наблюдаемости. |
| учебная демонстрация | ★★★★★ | легко понять Agent Основные понятия |

### 5. Комплексная оценка

**SimpleAgent Рейтинг:★★☆☆☆ (2/5)**

**Преимущества**：
- простая реализация, мало кода
- Быстрый отклик, подходит для быстрого прототипирования
- легко понять и отладить

**недостатки**：
- Нет инженерной устойчивости для сложного бизнеса
- Чёрный ящик, сложно вмешаться
- Сильное раздувание контекста
- Нет явного планирования

**Вывод**: SimpleAgent подходит как отладочный каркас иучебная демонстрация，для сложных сценариев — Plan-and-Solve или устойчивее.

### 6. Замеры производительности

В ходе реального тестирования мы получили следующие данные производительности:

- **Инструменты поиска информации**: Среднее время ответа составляет ок. 0.5-2 с(зависит от сети)
- **инструменты обработки данных**: Локальная обработка, время ответа < 0.01 Второй
- **Инструмент создания отчетов**: Локальная обработка, время ответа < 0.01 Второй
- **Полный процесс анализа**: 3 Анализ конкурентной продукции требует примерно 20-60 с（быстрее Plan-and-Solve, но меньше прозрачности плана）

**Время тестирования**: 2026-04-09

**иллюстрировать**:SimpleAgent реагирует быстрее,в сложных сценариях — потеря контекста или галлюцинации.

# Часть 7: Резюме и перспективы

## Часть 7: Резюме и перспективы проекта

### 1. Реализованные функции

Этот проект основан на **Hello Agents** Фреймворк успешно реализует анализ конкурентной продукции. Агент, основные результаты включают в себя:

**1. Создание основной инструментальной цепочки**
- ✅ **CompetitiveInfoSearchTool**: на основе Tavily API Настоящий инструмент поиска, который может динамически собирать информацию о конкурентных продуктах.
- ✅ **DataProcessorTool**: каркас (PoC)
- ✅ **ReportGeneratorTool**: каркас (PoC)

**2. SimpleAgent выполнить**
- ✅ Agent на `SimpleAgent`, ReAct
- ✅ Промпт с правилами вызова инструментов
- ✅ Регистрация и автовызов инструментов
- ✅ Цепочка: поиск → обработка → отчёт

**3. инженерная практика**
- ✅ Конфигурация переменной среды (.окр) управление API Keys
- ✅ Защитное программирование: включены инструменты поиска try-except Обработка исключений
- ✅ Агентская поддержка: адаптирована к домашней сетевой среде HTTPS_PROXY Конфигурация

### 2. Вызовы и решения

| Вызов | Решение | Статус |
|------|---------|------|
| **Импорт Tool** | `Tool` + `get_parameters()` | ✅ Решено |
| **Пустые параметры** | Промпт: извлечение названия | ✅ Решено |
| **сеть SSL ошибка** | обработка исключений и резерв，Конфигурацияпрокси | ✅ Решено |
| **Набухание контекста** |Архитектура SimpleAgent имеет неотъемлемые ограничения, которые требуютсценарийПерейти на план-and-Solve | ⚠️Известные ограничения|
| **инструмент"Финт"** | DataProcessorTool и ReportGeneratorTool фиксированная строка, Реальные данные не обработаны| ⚠️Нуждается в улучшении|

### 3. Ключевые извлеченные уроки

**1. Agent Проектные точки**
- Слова системных подсказок должны быть достаточно подробными и давать четкие указания. LLM как вызывать инструменты
- Названия параметров инструмента должны быть краткими и понятными (например, `name` Сравнивать `product_name` более восприимчив кLLM Understanding)- Необходимо выполнить защитное программирование, сетевые запросы могут завершиться неудачей в любой момент.

**2. SimpleAgent ограничения**
-Fit 3до конкурентныйбыстроАнализ
-Не подходит для более сложныхШагиЗадача (легко теряется контекст)- Чёрный ящик, сложно вмешатьсяи отладка**3. Принципы проектирования инструментальной цепочки**
- Инструменты должны фактически обрабатывать данные, а не возвращать фиксированный текст.
-каждыйшт.Инструменты должны делать только одно и поддерживать единственную ответственность-Входные и выходные данные инструмента должны быть проверяемыми и проверяемыми###IV.направления развития

**Краткосрочные улучшения (1-2 неделя)**
- [ ] **Реальная обработка данных**：позволитьИстинный синтаксический анализ DataProcessorToolпоисквозвращенный текст,извлечениеСтруктурированная информация- [ ] **Аутентичный отчетгенерация**：позволить ReportGeneratorTool Формируйте отчеты на основе структурированных данных
- [ ] **Механизм повтора**:использоватьбиблиотека упорства - этопоискСервис - Добавить - Автоматическая повторная попытка- [ ] **Механизм кэширования**: Кэшируйте результаты поиска, чтобы избежать повторных вызовов. API

**Среднесрочные улучшения (1 месяцев)**
- [ ] **Множественный внутренний поиск**:поддерживать DuckDuckGo как Tavily бесплатная альтернатива
- [ ] **РезультатСтойкость**Изложить п. 3.7. АнализРезультатСохранение в базу данных для поддержки исторических запросов- [ ] **визуализация**:использовать matplotlib/plotly Создание контрастных радиолокационных диаграмм и гистограмм
- [ ] **ПартияАнализ**: поддержка из CSV-файла/Импорт ExcelконкурентСписки для дозированияАнализ

**Долгосрочные улучшения (3 месяцев)**
- [ ] **Web интерфейс**:использовать Gradio/Streamlit Создавайте удобные интерфейсы
- [ ] **инкрементальное обновление**：регулярный мониторинг конкурентов，автообнаружение изменений
- [ ] **Мультимодальная поддержка**: Анализируйте скриншоты, рекламные видеоролики и т. д. конкурирующих продуктов.
- [ ] **Возможности совместной работы**：совместная работа командыАнализРезультат、комментарии

###V.итоговая оценка

**SimpleAgent в этом проекте**：
- ✅ **ОбучениеценностьВысокое**: Код лаконичен и прост для понимания АгентОсновные принципы
- ✅ **быстроПодтверждение**: Подходит для быстрого прототипирования и обучающих демонстраций.
- ⚠️ **Ограничения производства**: Не обладает инженерной устойчивостью сложного бизнеса**рекомендуемые действия**：
1. краткосрочно: Улучшение DataProcessorToolи ReportGeneratorTool: цепочка инструментов действительно работает
2. Среднесрочная перспектива: Сравнительный опыт PlanAndSolveAgent, поймите разницу между двумя парадигмами
3. Долгосрочная перспектива: выберите подходящую архитектуру, основанную на потребностях бизнеса в продуктизации.

---

**Срок завершения проекта**: 2026-04-09  
**автор**: czxgg0630  
**GitHub**: https://github.com/czxgg0630